In [ ]:
# importing saved model form drive

In [10]:
!pip install -U transformers accelerate peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 24.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.17.0
    Uninstalling transformers-5.17.0:
      Successfully uninstalled transformers-5.17.0
  Attempting uninstall: peft
    Found existing installation: peft 0.21.0
    Uninstalling peft-0.21.0:
      Successfully uninstalled peft-0.21.0


In [7]:
!pip install -U "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.1 MB/s eta 0:00:00


In [4]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import os

model_path = "/content/drive/MyDrive/banking-chatbot-lora4"

print(os.listdir(model_path))

['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json', 'training_args.bin']


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel




In [7]:
base_model_name = "Qwen/Qwen2.5-3B-Instruct"
# model path
model_path = "/content/drive/MyDrive/banking-chatbot-lora4"
adapter_path = model_path

In [8]:

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)



In [9]:
tokenizer = AutoTokenizer.from_pretrained(adapter_path)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    adapter_path
)

model.eval()

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048)
        (layers): ModuleList(
          (0-35): 36 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Li

In [11]:
# evaluation using langsmith

In [10]:
test_data = [
    {
        "question": "What documents are required to open a savings account?",
        "expected_answer": "Customers generally need identity proof, address proof, PAN or Form 60, and other documents required by the bank."
    },
    {
        "question": "What is NEFT?",
        "expected_answer": "NEFT is a banking payment system used to electronically transfer money from one bank account to another."
    },
    {
        "question": "What is RTGS?",
        "expected_answer": "RTGS is a payment system that transfers funds individually and settles transactions in real time."
    },
    {
        "question": "What should I do if my debit card is lost?",
        "expected_answer": "The customer should immediately block or report the lost card through the bank's available channels and request a replacement."
    },
    {
        "question": "How can I reset my banking password?",
        "expected_answer": "The customer can use the bank's password reset option and follow the required identity verification steps."
    }
]

In [11]:
# generating responses for every qiestion
def generate_response(question):

    messages = [
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.2,
            do_sample=True
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return response.strip()

In [12]:
print(generate_response(
    "What is the difference between NEFT and RTGS?"
))

NEFT (National Electronic Funds Transfer) and RTGS (Real Time Gross Settlement) are both systems used in India for transferring money electronically, but they differ in several aspects:

1. **Nature of Transactions**:
   - **NEFT**: NEFT is an opt-in system that allows users to transfer funds between bank accounts at different banks. It processes transactions in batches throughout the day.
   - **RTGS**: RTGS is a real-time, one-to-one transaction system that settles funds immediately after the transaction is completed.

2. **Processing Speed**:
   - **NEFT**: Transactions are processed in batches and can take up to 2-3 business days to clear. This makes it less suitable for urgent transfers.
   - **


In [10]:
# for whole question set
for item in test_data:

    response = generate_response(item["question"])

    item["model_answer"] = response

    print("Question:", item["question"])
    print("Expected:", item["expected_answer"])
    print("Model:", response)
    print("-" * 80)

Question: What documents are required to open a savings account?
Expected: Customers generally need identity proof, address proof, PAN or Form 60, and other documents required by the bank.
Model: The specific documents required to open a savings account can vary depending on the bank or financial institution you choose and your location. However, here are some common requirements that many banks might ask for:

1. **Identification Document**: This is usually a form of government-issued ID such as a passport, national ID card, or driver's license. In some countries, a utility bill (like an electricity or water bill) may also be accepted.

2. **Proof of Address**: This could be a utility bill, a recent bank statement, or another document showing your current address.

3. **Bank Account Number**: If you already have a bank account, you might need to provide this information.

4. **Account Opening Form**: Many banks will require you
---------------------------------------------------------

In [13]:
import os
from getpass import getpass

os.environ["LANGSMITH_API_KEY"] = getpass("Enter LangSmith API key: ")

Enter LangSmith API key: ··········


In [14]:
from langsmith import Client

client = Client()

In [14]:
# dataset creation
dataset = client.create_dataset(
    dataset_name="Banking AI Assistant Evaluation"
)

In [15]:
# uploading the test questions expected answers
for item in test_data:
    client.create_example(
        inputs={
            "question": item["question"]
        },
        outputs={
            "answer": item["expected_answer"]
        },
        dataset_id=dataset.id
    )

In [25]:
# llm as judge -->groq
!pip install -q langchain-groq langchain


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.1 MB/s eta 0:00:00


In [27]:
os.environ["GROQ_API_KEY"] = getpass(
    "Enter groq API key: "
)

Enter groq API key: ··········


In [33]:
from langchain_groq import ChatGroq

judge_llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

In [34]:
# evaluation by this groq provider
from langchain_core.prompts import ChatPromptTemplate

evaluation_prompt = ChatPromptTemplate.from_template("""
You are evaluating a banking AI assistant.

Question:
{question}

Expected Answer:
{expected_answer}

Model Answer:
{model_answer}

Evaluate whether the model answer correctly answers the question.

Give a score from 0 to 1:

1.0 = completely correct
0.5 = partially correct
0.0 = incorrect

Return only the score and a short explanation.
""")

In [35]:
# finetuned model response-->
def target(inputs):
    question = inputs["question"]

    response = generate_response(question)

    return {
        "answer": response
    }

In [36]:
# actual llm evalutor
def correctness_evaluator(inputs, outputs, reference_outputs):
    question = inputs["question"]
    model_answer = outputs["answer"]
    expected_answer = reference_outputs["answer"]

    prompt = f"""
You are evaluating a banking AI assistant.

Question:
{question}

Expected Answer:
{expected_answer}

Model Answer:
{model_answer}

Score the model answer:

1.0 = Completely correct
0.5 = Partially correct
0.0 = Incorrect

Consider whether the answer is factually correct and answers the question.

Return only:
Score: <number>
Reason: <short explanation>
"""

    result = judge_llm.invoke(prompt)

    text = result.content

    # Extract score
    import re

    match = re.search(r"Score:\s*(0(?:\.\d+)?|1(?:\.0+)?)", text)

    if match:
        score = float(match.group(1))
    else:
        score = 0.0

    return {
        "key": "correctness",
        "score": score,
        "comment": text
    }

In [37]:
results = client.evaluate(
    target,
    data="Banking AI Assistant Evaluation",
    evaluators=[correctness_evaluator],
    experiment_prefix="banking-qlora-evaluation"
)

View the evaluation results for experiment: 'banking-qlora-evaluation-709b4d4b' at:
https://smith.langchain.com/o/7ed02631-ff1d-49ef-8f51-26f2f9ef8e4c/datasets/b00bafd0-8abb-40b6-842f-2f9296fffbaa/compare?selectedSessions=2bb86db5-e30b-47c3-ba1d-d934cdac525c




0it [00:00, ?it/s]